# Example 24 — Composite wall: interfaces and a taste of domain decomposition

Two slabs in series ($k_1 = 1$ for $x<0.5$, $k_2 = 5$ for $x>0.5$), fixed face
temperatures:
$$\frac{d}{dx}\Big(k(x)\frac{dT}{dx}\Big) = 0,\qquad T(0)=1,\ T(1)=0.$$
Exact: piecewise linear with a **kink** at the interface — temperature continuous, flux
$k\,T'$ continuous, slope *dis*continuous (ratio 5:1). The thermal-circuit answer:
$q = 1/(R_1+R_2)$, interface temperature $T_i = 0.1667$.

**Why one network struggles:** a smooth tanh net can't put a corner in $T$ — the same
smoothness that broke the square wave (Ex. 1) and shocks (Ex. 10). **The fix is Example 9
in miniature:** one small net per layer, glued by interface conditions — a two-subdomain
decomposition where the interface constraints are the physics itself.

Verified: L2 ≈ 1.3e-05, interface temperature 0.1667 vs exact 0.1667, ~6 s on CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

K1, K2, XI = 1.0, 5.0, 0.5
q  = 1.0/(XI/K1 + (1-XI)/K2)         # thermal circuit: series resistances
Ti = 1 - q*XI/K1                      # exact interface temperature
def T_exactf(x):
    return np.where(x < XI, 1 - q*x/K1, q*(1-x)/K2)

n1 = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,1)).to(device)
n2 = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,1)).to(device)
T1 = lambda x: 1 + x*n1(x)            # T1(0)=1 hard
T2 = lambda x: (1-x)*n2(x)            # T2(1)=0 hard
opt = torch.optim.Adam(list(n1.parameters())+list(n2.parameters()), 2e-3)
xi = torch.full((1,1), XI, device=device)

t0 = time.perf_counter()
for e in range(3000):
    opt.zero_grad()
    xa = (torch.rand(256,1,device=device)*XI).requires_grad_(True)
    xb = (torch.rand(256,1,device=device)*(1-XI)+XI).requires_grad_(True)
    r1 = g1(g1(T1(xa),xa),xa)                       # k const per layer: T'' = 0
    r2 = g1(g1(T2(xb),xb),xb)
    xm = xi.clone().requires_grad_(True)
    t1, t2 = T1(xm), T2(xm)
    d1, d2 = g1(t1,xm), g1(t2,xm)
    loss = (r1**2).mean() + (r2**2).mean() \
         + 10*((t1-t2)**2).sum() \
         + 10*((K1*d1 - K2*d2)**2).sum()            # flux continuity: the interface physics
    loss.backward(); opt.step()
if device.type=='cuda': torch.cuda.synchronize()
print(f'training: {time.perf_counter()-t0:.0f} s')
with torch.no_grad():
    print(f'interface T: PINN {float(T1(xi)):.4f}   exact {Ti:.4f}')

x1 = torch.linspace(0, XI, 200, device=device).reshape(-1,1)
x2 = torch.linspace(XI, 1, 200, device=device).reshape(-1,1)
with torch.no_grad():
    p1 = T1(x1).cpu().numpy().ravel(); p2 = T2(x2).cpu().numpy().ravel()
xg = np.linspace(0,1,400)
plt.figure(figsize=(8,4))
plt.plot(xg, T_exactf(xg), 'g', lw=2.4, label='exact (piecewise linear)')
plt.plot(x1.cpu(), p1, 'r--', lw=1.6, label='PINN layer 1 (k=1)')
plt.plot(x2.cpu(), p2, 'm--', lw=1.6, label='PINN layer 2 (k=5)')
plt.axvline(XI, color='gray', ls=':', lw=1)
plt.xlabel('x'); plt.ylabel('T'); plt.legend(); plt.grid(alpha=.3)
plt.title(f'Composite wall: slope ratio 5:1 at the interface;  T_i = {float(T1(xi)):.4f}')
plt.tight_layout(); plt.show()

## Observations
- **Interfaces = physics as loss terms.** Temperature continuity and flux continuity
  $k_1 T_1' = k_2 T_2'$ are exactly what a conjugate-heat-transfer solver enforces at a
  solid–solid boundary; here they're two penalty terms at one point.
- **Two nets beat one.** A single smooth network smears the kink (try it — swap in one net
  and watch the interface flux error); per-layer networks make the corner representable.
  This is FBPINN/XPINN (Example 9) at its smallest possible scale.
- **The thermal circuit falls out:** heat flux $q = \Delta T/(R_1+R_2)$ and the interface
  temperature match the series-resistance hand calculation to 4 digits.

**Try:** three layers; temperature-dependent $k(T)$ in one layer (nonlinear); an interfacial
contact resistance ($T$ jump proportional to flux — one changed line in the interface loss).